# Windolf et al. (2025), Extended Data Fig. 3, on dandiset 001290: the IBL preprocessing variant of step 1

Paper: Windolf, Yu, Paulk et al. (2025), Nature Methods, doi:10.1038/s41592-025-02614-5. Dataset:
DANDI:001290, version 0.260827.1449, as in `reproduce_windolf_2025.ipynb`, whose header gives the
claims, the pinned version and the text the claims file was written from.

Step 1 of the claims file's pipeline has two readings: (i) the IBL preprocessing followed by
DARTsort detection, denoising and localization, and (ii) SpikeInterface detection and monopolar
triangulation after a bandpass and a median reference. Reading (i) could not be run (DARTsort on CPU
is far too slow; see the main notebook's header). This notebook runs the part of it that can run:
the IBL preprocessing as the Methods describe it ("high-pass filtering, analog-to-digital
converter offset correction, dead and noisy channel detection, and spatial high-pass filtering,
was reproduced using modular components available in the SpikeInterface framework", Preprocessing
of AP band data, paragraph 1), then the detection and localization of reading (ii). It is a
partial check of how much the preprocessing of step 1 changes the four results; it is not reading
(i), as the detection, denoising and localization stay those of reading (ii). The tests are those
of the main notebook. It takes about 30 minutes.

In [ ]:
import os
for v in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMBA_NUM_THREADS"]:
    os.environ[v] = "1"
import time
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pulsation as P

URL = P.asset_url()
f = P.open_file(URL)
ap = f[P.AP_PATH]
fs_ap = float(ap["starting_time"].attrs["rate"])
T_END = ap["data"].shape[0] / fs_ap
locs, shift = P.probe_geometry(f)
YLO, YHI = float(locs[:, 1].min()), float(locs[:, 1].max())
print(f"DANDI:{P.DANDISET_ID} version {P.DANDISET_VERSION}, {P.ASSET_PATH}: AP {ap['data'].shape} at {fs_ap} Hz, {T_END:.3f} s")

## Preprocessing, detection and localization

Bad channels are found with SpikeInterface's `detect_bad_channels` (the IBL method,
coherence+psd, its defaults) on 12 blocks spread over the recording, each channel taking the label
it gets in most blocks. Each block is then bandpass filtered (300 to 6,000 Hz), phase shifted by
the file's `inter_sample_shift`, its dead and noisy channels interpolated and the channels outside
the brain removed, and spatially high-pass filtered; detection and localization follow as in
reading (ii). This takes about 20 minutes.

In [ ]:
t0 = time.time()
ibl = P.ibl_bad_channels(URL)
noise = P.noise_levels(URL, pipeline="ibl", ibl=ibl)
sp = P.detect_spikes(URL, noise=noise, n_workers=4, pipeline="ibl", ibl=ibl)
print(f"bad channels, noise and detection took {(time.time() - t0) / 60:.1f} min")

In [ ]:
names = ["good", "dead", "noise", "out"]
print("channel labels (majority over 12 blocks):", {n: int((ibl["labels"] == n).sum()) for n in names})
for n in names[1:]:
    ch = np.where(ibl["labels"] == n)[0]
    print(f"  {n}: channels {ch.tolist()}, y {locs[ch, 1].tolist()} um, blocks with this label {ibl['counts'][names.index(n)][ch].tolist()}")
print("channels with a label other than their majority in some block:", int((ibl["counts"].max(0) < ibl["counts"].sum(0)).sum()))
kept = ibl["labels"] != "out"
print("noise level after preprocessing (uV, sd from MAD), over the", int(kept.sum()), "channels kept: median",
      round(float(np.median(noise)), 2), "range", round(float(noise.min()), 2), "to", round(float(noise.max()), 2))
n = len(sp["t"])
print("spikes detected:", n, "| per second", round(n / T_END, 1))
print("time from", round(sp["t"].min(), 3), "to", round(sp["t"].max(), 3), "s")
for k, unit in [("amp", "uV"), ("x", "um"), ("y", "um"), ("z", "um")]:
    q = np.percentile(sp[k], [0, 1, 50, 99, 100])
    print(f"{k} ({unit}): min {q[0]:.1f}, 1% {q[1]:.1f}, median {q[2]:.1f}, 99% {q[3]:.1f}, max {q[4]:.1f}")
out = (sp["y"] < YLO) | (sp["y"] > YHI)
print("spikes localized beyond the probe's ends:", int(out.sum()), f"({out.mean():.1%})")
hd, he = np.histogram(sp["y"], bins=np.arange(-200, 4001, 400))
print("spikes per 400 um of depth:", dict(zip(he[:-1].astype(int).tolist(), hd.tolist())))

## edf3a-ap-dredge-slow-trend

> In the fast-motion macaque recording, DREDge's spike-based (AP) motion estimate follows the drift
> visible in the unregistered spike raster.

Registration readings (a) nonrigid and (b) rigid; mean template correlation against the
unregistered raster, on the unsmoothed and the smoothed raster, as in the main notebook.

In [ ]:
reg, yreg, tc_reg, tc_reg_s = {}, {}, {}, {}
lo0, hi0 = P.probe_extent(None, YLO, YHI)
tc_unreg, corr_unreg = P.template_correlation(sp["t"], sp["y"], sp["amp"], lo0, hi0, t_max=T_END)
tc_unreg_s, _ = P.template_correlation(sp["t"], sp["y"], sp["amp"], lo0, hi0, t_max=T_END, smooth=True)
print(f"unregistered: mean template correlation {tc_unreg:.4f} (smoothed raster {tc_unreg_s:.4f})")
for name, rigid in [("a_nonrigid", False), ("b_rigid", True)]:
    me = reg[name] = P.register(sp, rigid=rigid)
    yreg[name] = me.correct_s(sp["t"], sp["y"])
    lo, hi = P.probe_extent(me, YLO, YHI)
    tc_reg[name], corr = P.template_correlation(sp["t"], yreg[name], sp["amp"], lo, hi, t_max=T_END)
    tc_reg_s[name], _ = P.template_correlation(sp["t"], yreg[name], sp["amp"], lo, hi, t_max=T_END, smooth=True)
    d = me.displacement
    print(f"{name}: mean template correlation {tc_reg[name]:.4f} (change {tc_reg[name] - tc_unreg:+.4f}), time bins improved"
          f" {np.nanmean(corr > corr_unreg):.1%}; smoothed raster {tc_reg_s[name]:.4f} (change {tc_reg_s[name] - tc_unreg_s:+.4f})")
    print(f"   displacement range {d.min():.1f} to {d.max():.1f} um, sd {d.std():.1f} um; change between 1 s bins median"
          f" {np.median(np.abs(np.diff(d, axis=-1))):.1f} um")
dr = reg["b_rigid"].displacement
print("rigid displacement, mean per 60 s (um):", np.round([dr[i:i + 60].mean() for i in range(0, len(dr), 60)], 1))

**Result, IBL preprocessing variant: the criterion's measure holds under both registration readings, as with reading (ii).** The mean template correlation rises from 0.2897 unregistered to 0.3300 with nonrigid and 0.3360 with rigid registration, and is higher in 75.2% and 76.0% of the 1 s bins; on the smoothed raster it rises from 0.6295 to 0.7108 and 0.7209. The rigid estimate ranges from -82.5 to 130.9 um (s.d. 44.2 um), and its 60 s means from -37.3 to 42.9 um. The preprocessing labels 10 channels dead (at intervals of 37 to 39 channels, in all 12 blocks), one noisy (channel 86) and 6 outside the brain at the top of the probe, and 359,760 spikes are detected.

## Fast motion: clustering and splines (pipeline step 3)

The same 32 combinations as in the main notebook.

In [ ]:
t0 = time.time()
combos = P.fast_motion_combos(sp, yreg, T_END)
print(len(combos), "combinations in", round((time.time() - t0) / 60, 1), "min")

## edf3b-hdbscan-clusters

> After AP DREDge registration, high-amplitude spikes form clusters that HDBSCAN isolates using
> their registered positions and amplitudes.

At least two clusters other than noise in every combination.

In [ ]:
print(f"{'reg':11s} {'cut':>4s} {'features':9s} {'mcs':>4s} {'spikes':>7s} {'clusters':>8s} {'noise':>6s}  sizes (largest first)")
n_ok = 0
for c in combos:
    if c["pen"]:
        continue
    k = len(c["sizes"])
    n_ok += k >= 2
    print(f"{c['reg']:11s} {c['frac']:4.2f} {c['feats']:9s} {c['mcs']:4d} {c['n_sel']:7d} {k:8d} {np.mean(c['labels'] < 0):6.1%}  {sorted(c['sizes'].tolist(), reverse=True)[:10]}")
print(f"combinations with at least two clusters: {n_ok} of {len(combos) // 2}")

**Result, IBL preprocessing variant: reproduced.** All 16 combinations return at least two clusters other than noise (9 to 35 clusters).

## edf3c-spline-subsecond

> A smoothing spline fit to the centered positions of the clustered spikes captures sub-second
> displacement left after AP DREDge registration.

The claims file's scrambled control (0.4 s bins, 100 permutations, 5% level), and the held-out
check of the main notebook with random halves of each cluster's spikes.

In [ ]:
rng_c = np.random.default_rng(20251)
rng_h = np.random.default_rng(20252)
N_PERM = 100
print(f"{'reg':11s} {'cut':>4s} {'features':9s} {'mcs':>4s} {'spline':9s} {'sd none':>8s} {'sd spline':>9s} {'scr 5%':>7s} {'pass':>5s} {'sd um':>6s} | spikes held out, both ways")
n_pass = 0
for c in combos:
    s_none, s_spl, scr = P.scrambled_test(c["spline"], c["tt"], c["cen"], c["lab"], T_END, N_PERM, rng_c)
    ok = s_spl < np.percentile(scr, 5)
    n_pass += ok
    hs = P.heldout_test(c["tt"], c["cen"], c["lab"], T_END, c["pen"], P.split_spikes(c["lab"], rng_h), N_PERM, rng_h)
    c.update(sd_none=s_none, sd_spline=s_spl, ho_sp=hs)
    tr = c["spline"](np.arange(0.0, T_END, 0.01))
    print(f"{c['reg']:11s} {c['frac']:4.2f} {c['feats']:9s} {c['mcs']:4d} {'penalized' if c['pen'] else 'lsq':9s} {s_none:8.2f} {s_spl:9.2f} {np.percentile(scr, 5):7.2f} {str(bool(ok)):>5s} {tr.std():6.1f} |"
          f" {hs[0]['sd_none']:.1f}->{hs[0]['sd_spline']:.1f}  {hs[1]['sd_none']:.1f}->{hs[1]['sd_spline']:.1f}")
print(f"combinations where the spline beats the 5th percentile of the scrambled control: {n_pass} of {len(combos)}")
for pen in (False, True):
    hh = [h for c in combos if c["pen"] == pen for h in c["ho_sp"]]
    red = np.array([1 - h["sd_spline"] / h["sd_none"] for h in hh])
    print(f"{'penalized' if pen else 'least-squares'} spline, spikes held out: sd lower with the spline in {(red > 0).sum()} of {len(hh)},"
          f" change from {-red.max():+.1%} to {-red.min():+.1%}, median {-np.median(red):+.1%}")
print("penalties chosen by GCV:", sorted({c['info']['lam'] for c in combos if c['pen']}))

**Result, IBL preprocessing variant: reproduced by the claims file's control, and the penalized spline still lowers the s.d. of held-out spikes.** The spline-corrected s.d. is below the 5th percentile of the scrambled control in 32 of 32 combinations. Fit on half of each cluster's spikes, the penalized spline lowers the s.d. of the other half in 32 of 32 cases, by 0.6% to 27.3% (median 5.9%); the least-squares spline does in 23 of 32, and raises it by up to 295.4% in others. GCV picks penalties of 0.01 to 1.

## edf3de-spline-improves-registration

> Correcting the AP-registered spikes with the spline estimate of sub-second displacement gives a
> more stable spike raster and tighter spike localizations than AP DREDge registration alone.

Panel d on 1 s bins (the outcome's measure), on the smoothed raster, and on the spikes left out of
the fit; panel e, the pooled within-cluster sd.

In [ ]:
print(f"{'reg':11s} {'cut':>4s} {'features':9s} {'mcs':>4s} {'spline':9s} {'TC dredge':>9s} {'TC +spl':>8s} {'d up':>5s} {'smooth dr':>9s} {'smooth +s':>9s} {'left-out dr':>11s} {'left-out +s':>11s} {'sd dredge':>9s} {'sd +spl':>8s}")
for c in combos:
    me, yr = reg[c["reg"]], yreg[c["reg"]]
    lo, hi = P.probe_extent(me, YLO, YHI)
    y2 = yr - c["spline"](sp["t"])
    lo2, hi2 = P.probe_extent(me, YLO, YHI, extra=c["spline"])
    tc2, _ = P.template_correlation(sp["t"], y2, sp["amp"], lo2, hi2, t_max=T_END)
    tcs2, _ = P.template_correlation(sp["t"], y2, sp["amp"], lo2, hi2, t_max=T_END, smooth=True)
    o = ~c["sel"]
    tco1, _ = P.template_correlation(sp["t"][o], yr[o], sp["amp"][o], lo, hi, t_max=T_END)
    tco2, _ = P.template_correlation(sp["t"][o], y2[o], sp["amp"][o], lo2, hi2, t_max=T_END)
    c.update(tc_spline=tc2, tcs_spline=tcs2, tco=(tco1, tco2))
    print(f"{c['reg']:11s} {c['frac']:4.2f} {c['feats']:9s} {c['mcs']:4d} {'penalized' if c['pen'] else 'lsq':9s} {tc_reg[c['reg']]:9.4f} {tc2:8.4f} {str(bool(tc2 > tc_reg[c['reg']])):>5s} {tc_reg_s[c['reg']]:9.4f} {tcs2:9.4f} {tco1:11.4f} {tco2:11.4f} {c['sd_none']:9.2f} {c['sd_spline']:8.2f}")
print(f"panel d, template correlation higher with the spline: {sum(c['tc_spline'] > tc_reg[c['reg']] for c in combos)} of {len(combos)}")
print(f"panel e, pooled within-cluster sd lower with the spline: {sum(c['sd_spline'] < c['sd_none'] for c in combos)} of {len(combos)}")
for feats in ("x,y,amp", "y,amp"):
    cc = [c for c in combos if c["feats"] == feats]
    for lab_, ch in [("1 s bins", [c["tc_spline"] - tc_reg[c["reg"]] for c in cc]),
                     ("smoothed raster", [c["tcs_spline"] - tc_reg_s[c["reg"]] for c in cc]),
                     ("spikes left out of the fit", [c["tco"][1] - c["tco"][0] for c in cc])]:
        ch = np.array(ch)
        print(f"features {feats}, {lab_}: change with the spline from {ch.min():+.4f} to {ch.max():+.4f}, higher in {(ch > 0).sum()} of {len(ch)}")
for frac in (0.10, 0.25):
    ch = np.array([c["tc_spline"] - tc_reg[c["reg"]] for c in combos if c["frac"] == frac])
    print(f"top {frac:.0%} amplitude cut, 1 s bins: template correlation change with the spline from {ch.min():+.4f} to {ch.max():+.4f}, higher in {(ch > 0).sum()} of {len(ch)}")

**Result, IBL preprocessing variant: partially reproduced, as with reading (ii), though the combinations that fail differ.** Panel e's s.d. is lower with the spline in 32 of 32 combinations; panel d's template correlation is higher in 22 of 32. Here it is higher in all 16 combinations with the top 10% amplitude cut, by +0.0674 to +0.0973, and in 6 of 16 with the top 25% cut (changes from -0.1058 to +0.0315); by features, in 12 of 16 with lateral position, depth and amplitude and 10 of 16 with depth and amplitude only. With reading (ii) all the failures were among the combinations without lateral position; here they are among those with the top 25% cut, so which readings fail depends on the preprocessing.

## Summary of this variant

| Claim | With the IBL preprocessing before the detection of reading (ii) |
|---|---|
| edf3a-ap-dredge-slow-trend | template correlation higher after registration under both readings |
| edf3b-hdbscan-clusters | at least two clusters in every combination |
| edf3c-spline-subsecond | the scrambled control passes in every combination; the penalized spline lowers the held-out s.d. in every case |
| edf3de-spline-improves-registration | panel e in every combination, panel d in 22 of 32 |

These are the outcomes reading (ii) gives in the main notebook. The variant does not replace
reading (i): DARTsort's detection, denoising and localization were not run.